# GOES Event: 30 March 2026 to 3 April 2026
In this Notebook, I am investigating the energetic particle event observed in the GOES data on 1 April 2026. This figure plots time vs flux from 6:00 on March 30 to 6:00 on April 03, 2026. It plots data from Parker Solar Probe, Solar Orbiter, SOHO STEREO-A and WIND for 100 keV electrons, 1 MeV electrons and >25 MeV protons/ions.

## 100 keV Electron panel
- PSP ISOIS-EPILO F (W3) 65.9 - 153.5 keV
- SOLO/EPT sun 0.0866 - 0.1314 MeV
- STEREO-A/SEPT sun 85-125 keV
- Wind/3DP omni 75.91 - 140.97 keV

## 1 MeV Electron panel
- PSP ISOIS-EPIHI-HET A (sun) 0.71 - 1.41 MeV
- SOLO/HET sun 0.45 - 2.40 MeV
- STEREO-A/HET 0.7 - 2.8 MeV
- SOHO/EPHIN 0.67 - 10.4 MeV

## >25 MeV Protons/Ions panel
- PSP ISOIS-EPHIH-HET A (sun) 26.9 - 38.1 MeV
- SOLO/HET sun 25.09 - 41.18 MeV
- STEREO-A/HET 26.3 - 40.5 MeV
- SOHO/ERNE/HED 25.0 - 40.0 MeV

The data was downloaded using the SERPENTINE data_loader tool (https://serpentine-h2020.eu).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cdflib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

In [ ]:
# ============================================================
# Settings
# ============================================================
RES   = "10min"
START = pd.Timestamp("2026-03-31 00:00")
END   = pd.Timestamp("2026-04-04 00:00")   # exclusive end: covers 30 Mar, 31 Mar, 1 Apr

# Replaces the hard-coded DAYS tuple; works across month boundaries
DAYS = pd.date_range(START.normalize(), END - pd.Timedelta(seconds=1), freq="D")
# -> 2026-03-30, 2026-03-31, 2026-04-01

DIR_100KEV = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\100keV_electrons") 
DIR_1MEV   = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\1MeV_electrons")
DIR_PROTON = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\25MeV_protons")

COLOURS = {"PSP": "C0", "SolO": "C1", "STEREO-A": "C2", "SOHO": "C3", "Wind": "C4"}

# Quality-flag masks: (variable, minimum value to keep) or None for no mask
QF_SOLO_EPT = ("QUALITY_FLAG_1", 3)   # your original 100 keV choice
QF_SOLO_HET_ELEC   = None             # e.g. ("QUALITY_FLAG_4", 3) once checked
QF_SOLO_HET_PROTON = None             # e.g. ("QUALITY_FLAG", 3) once checked


In [ ]:

# ============================================================
# Shared helpers
# ============================================================
def cdf_times(cdf, var):
    return pd.DatetimeIndex(cdflib.cdfepoch.to_datetime(cdf.varget(var)))

def resample(s):
    s = pd.Series(np.asarray(s, dtype=float), index=s.index).resample(RES).mean()
    return s.where(s > 0)                      # zeros can't go on a log axis

def band_average(flux, widths):
    """NaN-robust width-weighted mean. flux: (n_time, n_bins); widths: (n_bins,)."""
    flux = np.asarray(flux, dtype=float)
    flux = np.where(flux < 0, np.nan, flux)    # fill values (-1e31) and negatives
    valid = np.isfinite(flux)
    w = np.where(valid, np.asarray(widths, dtype=float), 0.0)
    num = np.sum(np.where(valid, flux, 0.0) * w, axis=1)
    den = w.sum(axis=1)
    out = np.full(len(flux), np.nan)
    np.divide(num, den, out=out, where=den > 0)
    return out

def day_files(data_dir, pattern, days=DAYS):
    """One file per day; if several versions exist, take the latest."""
    out = []
    for d in days:
        hits = sorted(data_dir.glob(pattern.format(d=d)))
        if not hits:
            raise FileNotFoundError(f"No match for {pattern.format(d=d)} in {data_dir}")
        out.append(hits[-1])
    return out

def load_band(data_dir, pattern, flux_var, epoch_var, idx, widths, qf=None):
    """Read selected energy bins from daily CDFs -> one width-weighted Series."""
    parts = []
    for f in day_files(data_dir, pattern):
        c = cdflib.CDF(str(f))
        t = cdf_times(c, epoch_var)
        flux = np.asarray(c.varget(flux_var), dtype=float)[:, idx]
        if qf is not None:
            flux[np.asarray(c.varget(qf[0])) < qf[1]] = np.nan
        parts.append(pd.Series(band_average(flux, widths), index=t))
    return pd.concat(parts).sort_index()

In [ ]:
# STEREO-A loader

def load_sept(files):
    bins = [f"{i:02d}" for i in range(2, 17)]
    names = (["jd", "year", "doy", "hour", "minute", "second"]
             + [f"flux_bin{b}" for b in bins]
             + [f"unc_bin{b}" for b in bins]
             + ["accum_time"])
    df = pd.concat(
        [pd.read_csv(f, comment="#", sep=r"\s+", header=None, names=names)
         for f in files],
        ignore_index=True,
    )
    df.index = pd.DatetimeIndex(
        pd.to_datetime(df["jd"], unit="D", origin="julian").dt.round("s"),
        name="time",
    )
    df = df.drop(columns=["jd", "year", "doy", "hour", "minute", "second"])
    df = df.replace([-9999.9, -9999], np.nan)
    return df.sort_index()

DIR_STA_SEPT = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\100keV_electrons")  # folder with the .dat files
DOYS_SEPT    = (89, 90, 91, 92, 93)

sept_files = day_files(
    DIR_STA_SEPT,
    "sept_ahead_ele_sun_2026_{d:03d}_1min_l2_v*.dat",
    days=DOYS_SEPT,
)
sept = load_sept(sept_files)
print(sept.shape)
print(sept.index[0], "->", sept.index[-1])
sept[["flux_bin06", "flux_bin07"]].describe()

In [ ]:
# ============================================================
# Panel 1: ~100 keV electrons
# ============================================================
def load_psp_100(files, wedge=3, bins=slice(8, 12)):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        t = cdf_times(c, "Epoch_ChanE")
        flux = c.varget("Electron_Flux_ChanE")[:, wedge, bins]
        width = (c.varget("Electron_ChanE_Energy_DELTAPLUS")[0, wedge, bins]
                 + c.varget("Electron_ChanE_Energy_DELTAMINUS")[0, wedge, bins])
        parts.append(pd.Series(band_average(flux, width) * 1e3, index=t))   # per keV -> per MeV
    return pd.concat(parts).sort_index()

def load_solo_100(files, bins=slice(14, 19), qf=QF_SOLO_EPT):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        t = cdf_times(c, "EPOCH_1")
        flux = c.varget("Electron_Flux")[:, bins].astype(float)
        if qf is not None:
            flux[np.asarray(c.varget(qf[0])) < qf[1]] = np.nan
        width = c.varget("Electron_Bins_Width")[bins]
        parts.append(pd.Series(band_average(flux, width), index=t))         # already per MeV
    return pd.concat(parts).sort_index()

def load_wind_100(files, channel=3):
    parts = []
    for f in files:
        c = cdflib.CDF(str(f))
        tt = c.varget("TIME").astype(float)
        ok = np.isfinite(tt) & (tt > 0)
        flux = c.varget("FLUX")[:, channel].astype(float)
        flux = np.where(flux < 0, np.nan, flux) * 1e6                       # per eV -> per MeV
        parts.append(pd.Series(flux[ok], index=pd.DatetimeIndex(pd.to_datetime(tt[ok], unit="s"))))
    return pd.concat(parts).sort_index()

def load_sept_100(sept, cols=("flux_bin06", "flux_bin07"), widths=(20, 20)):
    return pd.Series(band_average(sept[list(cols)].values, widths), index=sept.index)

def build_100keV(sept):
    psp  = load_psp_100 (day_files(DIR_100KEV, "psp_isois-epilo_l2-pe_{d:%Y%m%d}_v*.cdf"))
    solo = load_solo_100(day_files(DIR_100KEV, "solo_L2_epd-ept-sun-rates_{d:%Y%m%d}_V*.cdf"))
    wind = load_wind_100(day_files(DIR_100KEV, "wi_sfsp_3dp_{d:%Y%m%d}_v*.cdf"))
    sta  = load_sept_100(sept)
    series = {
        "PSP/EPI-Lo W3, 65.9-153.5 keV": (resample(psp),  "PSP"),
        "SolO/EPT sun, 86.6-131.4 keV":  (resample(solo), "SolO"),
        "STEREO-A/SEPT sun, 85-125 keV": (resample(sta),  "STEREO-A"),
        "Wind/3DP omni, 75.9-141.0 keV": (resample(wind), "Wind"),
    }
    #events = {
        #"STEREO-A": dict(onset="2022-03-28 11:34:51.974", peak_time="2022-03-28 12:28:51.974"),
        #"Wind":     dict(onset="2022-03-28 11:46:53.709", peak_time="2022-03-28 12:59:23.549"),
    #}
    events = {}
    return series, events
    

In [ ]:
# ============================================================
# Panel 2: ~1 MeV electrons
# ============================================================
# SOHO/EPHIN left out: the .rl2 download warned that the electron data can be unreliable
#def read_rl2(fn):
#    df = pd.read_csv(DIR_1MEV / fn, sep=r"\s+", header=None, engine="python")
#    t = (pd.to_datetime(df[0].astype(str), format="%Y")
#         + pd.to_timedelta(df[1] - 1, unit="D")
#         + pd.to_timedelta(df[2], unit="ms"))
#    return pd.Series(df[8].astype(float).values, index=pd.DatetimeIndex(t))

def build_1MeV():
    solo = load_band(DIR_1MEV, "solo_L2_epd-het-sun-rates_{d:%Y%m%d}_V*.cdf",
                     "Electron_Flux", "EPOCH_4", [0, 1], [0.5847, 1.348],
                     qf=QF_SOLO_HET_ELEC)    # 0.45-2.40 MeV
# STEREO-A HET: no data available for this period
#    sta  = load_band(DIR_1MEV, "sta_l1_het_202203{d}_v*.cdf",
#                     "Electron_Flux", "Epoch", [0, 1], [0.7, 1.4])          # 0.7-2.8 MeV
    # TODO: confirm bins/widths against Electrons_Inv_ENERGY_LABL (pending your output)
    psp  = load_band(DIR_1MEV, "psp_isois-epihi_l2-het-rates60_{d:%Y%m%d}_v*.cdf",
                     "A_Electrons_Flux", "Epoch", [3, 4, 5, 6], [0.1, 0.2, 0.2, 0.2])
#    eph  = pd.concat([read_rl2("epi22087.rl2"), read_rl2("epi22088.rl2")]).sort_index()
    return {
#        "SOHO/EPHIN E1300 0.67-10.4 MeV": (resample(eph),  "SOHO"),
        "SolO/HET sun 0.45-2.40 MeV":     (resample(solo), "SolO"),
#        "STEREO-A/HET 0.7-2.8 MeV":       (resample(sta),  "STEREO-A"),
        "PSP/HET A 0.7-1.4 MeV":          (resample(psp),  "PSP"),
    }

In [ ]:
# ============================================================
# Panel 3: >25 MeV protons
# ============================================================
def build_protons():
    psp  = load_band(DIR_PROTON, "psp_isois-epihi_l2-het-rates60_{d:%Y%m%d}_v*.cdf",
                     "A_H_Flux", "Epoch", [8, 9], [5.1, 6.1])               # 26.9-38.1 MeV
    solo = load_band(DIR_PROTON, "solo_L2_epd-het-sun-rates_{d:%Y%m%d}_V*.cdf",
                     "H_Flux", "EPOCH", list(range(19, 25)),
                     [2.11, 2.24, 2.53, 2.91, 3.02, 3.28],
                     qf=QF_SOLO_HET_PROTON)                                 # 25.09-41.18 MeV
    # STEREO-A HET: no data available for this period
    #sta  = load_band(DIR_PROTON, "sta_l1_het_202203{d}_v*.cdf",
    #                 "Proton_Flux", "Epoch", [5, 6, 7, 8], [3.4, 3.8, 2.4, 4.9])  # 26.3-40.5 MeV
    soho = load_band(DIR_PROTON, "soho_erne-hed_l2-1min_{d:%Y%m%d}_v*.cdf",
                     "PH", "Epoch", [3, 4], [7.0, 8.0])                     # 25-40 MeV
    return {
        "PSP/HET A 26.9-38.1 MeV":      (resample(psp),  "PSP"),
        "SolO/HET sun 25.09-41.18 MeV": (resample(solo), "SolO"),
        #"STEREO-A/HET 26.3-40.5 MeV":   (resample(sta),  "STEREO-A"),
        "SOHO/ERNE/HED 25.0-40.0 MeV":  (resample(soho), "SOHO"),
    }

In [ ]:
# ============================================================
# Build and plot
# ============================================================
# 'sept' must already exist (the DataFrame from your 100 keV notebook)
p100, events = build_100keV(sept)
p1m          = build_1MeV()
pp           = build_protons()

for name, d in [("100 keV", p100), ("1 MeV", p1m), (">25 MeV p", pp)]:
    for lab, (s, _) in d.items():
        s = s.loc[START:END]
        print(f"{name:10s} {lab:34s} {s.notna().sum():4d} valid / {len(s)} bins")

ylab_e = "Electron intensity\n" r"[cm$^{-2}$ s$^{-1}$ sr$^{-1}$ MeV$^{-1}$]"
ylab_p = "Proton intensity\n"   r"[cm$^{-2}$ s$^{-1}$ sr$^{-1}$ MeV$^{-1}$]"
panels = [(p100, ylab_e, "~100 keV electrons"),
          (p1m,  ylab_e, "~1 MeV electrons"),
          (pp,   ylab_p, ">25 MeV protons")]

fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True, layout="constrained")

for ax, (series, ylabel, title) in zip(axes, panels):
    for label, (s, sc) in series.items():
        s = s.loc[START:END]
        ax.step(s.index, s.where(s > 0), where="post", color=COLOURS[sc], label=label, lw=1.2)
    ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3, which="both")
    ax.text(0.01, 0.95, title, transform=ax.transAxes, va="top", fontweight="bold")

# onset / peak markers: top panel only
#for sc, ev in events.items():
#    axes[0].axvline(pd.Timestamp(ev["onset"]),     color=COLOURS[sc], ls="-", lw=1.2)
#    axes[0].axvline(pd.Timestamp(ev["peak_time"]), color=COLOURS[sc], ls=":", lw=1.5)
#axes[0].plot([], [], color="grey", ls="-", label="Onset time")
#axes[0].plot([], [], color="grey", ls=":", label="Peak flux")

for ax in axes:
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize=8)

axes[-1].set_xlim(START, END)
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=12))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M\n%d %b"))
axes[-1].set_xlabel("Time / Date in year {START.year}")
fig.align_ylabels(axes)
plt.show()

## GOES section
This code reads in the GOES data from "GOES-Primary RT Solar Particle and Electron Flux (5-minute)", and plots it in a separate panel. The GOES data is an integral flux in pfu (cm⁻² s⁻¹ sr⁻¹). The original data is differential intensity in  units are cm⁻² s⁻¹ sr⁻¹ MeV⁻¹. THey cannot be plotted on the same y-axis.

In [ ]:
# ============================================================
# Load GOES data and resample to 10-min
# ============================================================
import os

goes_path = os.path.join(
    r"C:\Users\pauls\SEPs\IMAP\data\GOES",
    "goes_primary_5min_2026-02_to_now.csv",
)
goes_channels = {            # column name : legend label
    "P10": "GOES P10 (>10 MeV protons)",
    "P30": "GOES P30 (>30 MeV protons)",
    "P50": "GOES P50 (>50 MeV protons)",
     # "E2_0": "GOES E>2 MeV electrons"
}

goes = pd.read_csv(goes_path, parse_dates=["Time"])
missing = [c for c in goes_channels if c not in goes.columns]
if missing:
    raise KeyError(f"Not found: {missing}. Columns are: {list(goes.columns)}")

goes["Time"] = pd.to_datetime(goes["Time"], utc=True)
if pd.Timestamp(START).tzinfo is None:
    goes["Time"] = goes["Time"].dt.tz_localize(None)
else:
    goes["Time"] = goes["Time"].dt.tz_convert(pd.Timestamp(START).tzinfo)

goes = goes.set_index("Time")[list(goes_channels)]
goes = goes.where(goes > 0)
goes10 = goes.resample("10min", closed="left", label="left").mean().loc[START:END]
print(goes10.notna().sum())

In [ ]:
# ============================================================
# Build and plot (existing code, with a 4th panel added)
# ============================================================
# ... build_100keV / build_1MeV / build_protons and the print loop unchanged ...

fig, axes = plt.subplots(4, 1, figsize=(13, 14), sharex=True, layout="constrained")

# The existing loop is unchanged: zip() stops after the first three axes
for ax, (series, ylabel, title) in zip(axes, panels):
    for label, (s, sc) in series.items():
        s = s.loc[START:END]
        ax.step(s.index, s.where(s > 0), where="post", color=COLOURS[sc], label=label, lw=1.2)
    ax.set_yscale("log")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3, which="both")
    ax.text(0.01, 0.95, title, transform=ax.transAxes, va="top", fontweight="bold")

# onset / peak markers: top panel only (unchanged)
#for sc, ev in events.items():
#    axes[0].axvline(pd.Timestamp(ev["onset"]),     color=COLOURS[sc], ls="-", lw=1.2)
#    axes[0].axvline(pd.Timestamp(ev["peak_time"]), color=COLOURS[sc], ls=":", lw=1.5)
#axes[0].plot([], [], color="grey", ls="-", label="Onset time")
#axes[0].plot([], [], color="grey", ls=":", label="Peak flux")

# ------------------------------------------------------------
# NEW: GOES panel (integral fluxes)
# ------------------------------------------------------------
ax_g = axes[3]
styles = {"P10": ("tab:green", "-"), "P30": ("k", "-"),
          "P50": ("tab:purple", "-"),  "E2_0": ("tab:red", "--")}
for col, lab in goes_channels.items():
    colr, ls = styles[col]
    ax_g.step(goes10.index, goes10[col], where="post", color=colr, ls=ls, lw=1.2, label=lab)
ax_g.set_yscale("log")
ax_g.set_ylabel("GOES integral flux\n" r"[cm$^{-2}$ s$^{-1}$ sr$^{-1}$]")
ax_g.grid(alpha=0.3, which="both")
ax_g.text(0.01, 0.95, "GOES integral fluxes (10-min mean)",
          transform=ax_g.transAxes, va="top", fontweight="bold")

for ax in axes:
    ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize=8)

axes[-1].set_xlim(START, END)
axes[-1].xaxis.set_major_locator(mdates.HourLocator(interval=12))
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M\n%d %b"))
axes[-1].set_xlabel(f"Time / Date in year {START.year}")
fig.align_ylabels(axes)
plt.show()

In [ ]:
from pathlib import Path

data_dir = Path(r"C:\Users\pauls\SEPs\IMAP\data\01042026\25MeV_s")   # change to your folder
files = sorted(data_dir.glob("*"))

print(len(files), "files")
for f in files:
    print(f.name)